In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist  # библиотека базы выборок Mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Reshape, Input

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))

batch_size = 100

input_img = Input((28, 28, 1))
x = Flatten()(input_img)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
encoded = Dense(49, activation='relu')(x)

d = Dense(64, activation='relu')(encoded)
d = Dense(28 * 28, activation='sigmoid')(d)
decoded = Reshape((28, 28, 1))(d)

autoencoder = keras.Model(input_img, decoded, name="autoencoder")
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

autoencoder.fit(x_train, x_train,
                epochs=20,
                batch_size=batch_size,
                shuffle=True)

n = 10

imgs = x_test[:n]
decoded_imgs = autoencoder.predict(x_test[:n], batch_size=n)

plt.figure(figsize=(n, 2))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(imgs[i].squeeze(), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    ax2 = plt.subplot(2, n, i + n + 1)
    plt.imshow(decoded_imgs[i].squeeze(), cmap='gray')
    ax2.get_xaxis().set_visible(False)
    ax2.get_yaxis().set_visible(False)

plt.show()

def plot_digits(*images):
    images = [x.squeeze() for x in images]
    n = images[0].shape[0]  # число изображений

    plt.figure(figsize=(n, len(images)))
    for j in range(n):
        for i in range(len(images)):
            ax = plt.subplot(len(images), n, i * n + j + 1)
            plt.imshow(images[i][j])
            plt.gray()
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)

    plt.show()


def plot_homotopy(frm, to, n=10, autoencoder=None):
    z = np.zeros(([n] + list(frm.shape)))
    for i, t in enumerate(np.linspace(0., 1., n)):
        z[i] = frm * (1 - t) + to * t  # Гомотопия по прямой
    if autoencoder:
        plot_digits(autoencoder.predict(z, batch_size=n))
    else:
        plot_digits(z)


frm, to = x_test[y_test == 5][1:3]
plot_homotopy(frm, to)
plot_homotopy(frm, to, autoencoder=autoencoder)

## Izah

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# TensorFlow-un artıq xəbərdarlıq mesajlarını gizlədir.
# DİQQƏT: TensorFlow import olunmazdan ƏVVƏL yazılmalıdır.


import numpy as np
# NumPy — massivlərlə işləmək üçün.
# Bu kodda:
# - np.reshape   → massivin formasını dəyişmək (28x28 → 28x28x1)
# - np.zeros     → boş (sıfırlarla dolu) massiv yaratmaq
# - np.linspace  → iki ədəd arasında bərabər məsafəli ədədlər yaratmaq
#                  (homotopiya üçün 0-dan 1-ə qədər 10 addım)


import matplotlib.pyplot as plt
# matplotlib — şəkilləri və qrafikləri göstərmək üçün.
# Bu kodda:
# - plt.figure   → yeni şəkil pəncərəsi (kətan) yaratmaq
# - plt.subplot  → pəncərəni cədvələ bölüb müəyyən xanaya şəkil qoymaq
# - plt.imshow   → massivi şəkil kimi göstərmək
# - plt.show     → hazırlanmış şəkli ekrana çıxarmaq


from tensorflow.keras.datasets import mnist  # библиотека базы выборок Mnist
# MNIST — əl ilə yazılmış rəqəmlərin (0-dan 9-a qədər) məşhur data bazası.
# (Rusca şərh: "MNIST nümunələr bazası kitabxanası".)
#
# Tərkibi:
#   - 60000 öyrətmə şəkli (train)
#   - 10000 test şəkli (test)
#   - hər şəkil 28x28 piksel, ağ-qara (boz tonlu)
#   - hər pikselin qiyməti 0-dan 255-ə qədərdir (0 = qara, 255 = ağ)


from tensorflow import keras
# Keras-ın özü. Bu kodda yalnız bir yerdə lazımdır: keras.Model(...)
# (modeli giriş və çıxışı göstərməklə qurmaq üçün).


from tensorflow.keras.layers import Dense, Flatten, Reshape, Input
# Keras qatları:
#
# Dense   → tam bağlı qat (hər giriş hər neyronla bağlıdır).
# Flatten → çoxölçülü massivi 1 ölçülü vektora açır: (28, 28, 1) → (784,)
# Reshape → əksinə, vektoru yenidən istənilən formaya qaytarır: (784,) → (28, 28, 1)
# Input   → modelin giriş formasını elan edir.
#
# DİQQƏT: bu kodda rekurrent qat (GRU/LSTM) YOXDUR. Yalnız Dense qatları var.


# =====================================================================
# BU KOD NƏDİR? — AVTOENKODER (Autoencoder)
# =====================================================================
#
# Avtoenkoder — şəkli əvvəlcə KİÇİK bir koda SIXAN, sonra həmin koddan
# şəkli YENİDƏN BƏRPA etməyə çalışan neyron şəbəkəsidir.
#
#   şəkil (784 piksel) ──► [ENCODER] ──► kod (49 ədəd) ──► [DECODER] ──► bərpa olunmuş şəkil (784 piksel)
#
# Model özünə verilən şəkli çıxışda təkrar almağa çalışır. Amma arada
# "dar keçid" (49 neyron) var, ona görə model bütün pikselləri əzbərləyə
# bilmir və yalnız ƏN VACİB məlumatı saxlamağı öyrənir.
#
# Bu necə mümkündür? Rəqəm şəkilləri təsadüfi piksellər yığını deyil:
# hər rəqəmin bir forması var (xətt, qövs, dairə). Model 784 pikseli
# yox, bu "formaları" 49 ədədlə təsvir etməyi öyrənir.
#
# ƏVVƏLKİ İKİ KODLA MÜQAYİSƏ:
#
#                    Sentiment kodu          Zaman seriyası kodu     Bu kod (avtoenkoder)
# Məsələ:            təsnifat (2 sinif)      regressiya (1 ədəd)     bərpa (784 ədəd)
# Giriş (X):         söz nömrələri           6x6 matris              28x28 şəkil
# Cavab (Y):         etiket [1,0] / [0,1]    ortadakı nöqtə          ELƏ ŞƏKLİN ÖZÜ (Y = X)
# Qat növü:          Embedding + GRU         Bidirectional GRU       yalnız Dense
# Çıxış aktivasiyası: softmax                linear                  sigmoid
# Loss:              categorical_crossentropy mean_squared_error     mean_squared_error
# Modelin qurulması: Sequential              Sequential              Functional API (aşağıda)
# Etiket lazımdırmı: bəli                    bəli                    XEYR (öz-özünə öyrənmə)
#
# ƏSAS FƏRQ: bu məsələdə cavab ayrıca etiket deyil. Modelə "bu 7-dir"
# demirik. Cavab elə girişin özüdür. Ona görə rəqəmin sinfi (y_train)
# öyrətmədə İSTİFADƏ OLUNMUR.


# ---------------------------------------------------------------------
# 1) DATANIN YÜKLƏNMƏSİ VƏ HAZIRLANMASI
# ---------------------------------------------------------------------

(x_train, y_train), (x_test, y_test) = mnist.load_data()
# MNIST datasını yükləyir (ilk işlətmədə internetdən endirir və kompüterdə
# saxlayır, sonra yenidən endirmir).
#
# Dörd massiv qaytarır:
#   x_train → öyrətmə şəkilləri, forma (60000, 28, 28), tip uint8 (0..255)
#   y_train → öyrətmə şəkillərinin rəqəmləri, forma (60000,), məsələn 5, 0, 4, ...
#   x_test  → test şəkilləri,     forma (10000, 28, 28)
#   y_test  → test rəqəmləri,     forma (10000,)
#
# Bu kodda y_train heç yerdə istifadə olunmur. y_test isə yalnız sonda,
# "5" rəqəmli şəkilləri seçmək üçün lazım olur.

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255
# (Rusca şərh: "giriş datasının standartlaşdırılması".)
# Piksel qiymətlərini 0..255 aralığından 0..1 aralığına gətirir.
#
# Niyə? Neyron şəbəkələri kiçik ədədlərlə daha yaxşı öyrənir. Əlavə səbəb:
# çıxış qatında sigmoid var və o yalnız 0..1 arası ədəd verə bilir. Hədəf
# (yəni şəklin özü) də həmin aralıqda olmalıdır ki, model ona yaxınlaşa bilsin.
#
# DİQQƏT: bölmədən sonra massivin tipi float64 olur. Hər birinin yaddaşı
# təxminən: 60000 × 784 × 8 bayt ≈ 376 MB (yalnız x_train). Keras bunu
# öyrətmədə özü float32-yə çevirir.

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))
# Hər şəklin sonuna "kanal" ölçüsü əlavə edir:
#   (60000, 28, 28)  →  (60000, 28, 28, 1)
#   (10000, 28, 28)  →  (10000, 28, 28, 1)
#
# Son rəqəm 1 → şəkil tək kanallıdır (boz ton). Rəngli şəkildə 3 olardı (RGB).
# Elementlərin sayı dəyişmir, yalnız forma dəyişir (60000×28×28×1 eynidir).
# Kanal ölçüsü Keras-ın şəkil qatları üçün standart formatdır.
# Bu kodda Dense istifadə olunduğu üçün məcburi deyil (Flatten onsuz da
# açacaq), amma qaydaya uyğundur.

batch_size = 100
# Bir addımda modelə neçə şəkil veriləcək.
# 60000 / 100 = 600 addım bir epoch-da.


# ---------------------------------------------------------------------
# 2) MODELİN QURULMASI (Functional API)
# ---------------------------------------------------------------------
#
# Əvvəlki kodlarda model = Sequential(); model.add(...) yazırdıq. Burada
# başqa üsul var: HƏR QAT ÖZÜNDƏN ƏVVƏLKİ NƏTİCƏYƏ "FUNKSİYA KİMİ" TƏTBİQ
# OLUNUR. Məsələn, Dense(128)(x) → "x-i Dense(128) qatından keçir".
# Bu üsul daha sərbəstdir (budaqlanma, birdən çox giriş/çıxış mümkündür).
# Burada qatlar yenə də ardıcıl düzülüb, ona görə Sequential də işləyərdi.
#
# Bütün arxitektura:
#
#   Giriş       Flatten     Dense      Dense      KOD       Dense      Dense       Reshape
#  (28,28,1) ─► (784,) ─► (128,) ─► (64,) ─► [ (49,) ] ─► (64,) ─► (784,) ─► (28,28,1)
#   └────────────────── ENCODER ──────────────────┘ └──────── DECODER ────────┘
#
# Sıxılma nisbəti: 784 / 49 = 16 dəfə.

input_img = Input((28, 28, 1))
# Giriş qatı: forma (28, 28, 1). Batch ölçüsü yazılmır.
# input_img — bu real data deyil, "buraya şəkil gələcək" yerini tutan simvolik obyektdir.

x = Flatten()(input_img)
# Şəkli düzəldir: (28, 28, 1) → (784,)
# Çünki Dense qat yalnız 1 ölçülü vektorla işləyir. Şəkil piksel-piksel
# bir sətrə yığılır (28 × 28 × 1 = 784).
# Parametr: yoxdur.
#
# DİQQƏT: burada x adlı dəyişən dataya yox, "şəbəkənin cari nöqtəsinə"
# aiddir. Bu, öyrətmə massivi x_train ilə qarışdırılmamalıdır.

x = Dense(128, activation='relu')(x)
# Birinci sıxma qatı: 784 → 128, aktivasiya ReLU.
# ReLU: mənfi ədədi 0 edir, müsbəti olduğu kimi saxlayır → max(0, ədəd).
# Forma: (784,) → (128,)
# Parametr sayı: 784 × 128 + 128 = 100480

x = Dense(64, activation='relu')(x)
# İkinci sıxma qatı: 128 → 64.
# Forma: (128,) → (64,)
# Parametr sayı: 128 × 64 + 64 = 8256

encoded = Dense(49, activation='relu')(x)
# DARBOĞAZ (bottleneck) qatı: 64 → 49. Bu avtoenkoderin ən kritik yeridir.
# Şəklin bütün məlumatı burada cəmi 49 ədədə sığdırılır. Bu 49 ədədlik
# vektora "gizli kod" (latent code / latent vector) deyilir.
# Forma: (64,) → (49,)
# Parametr sayı: 64 × 49 + 49 = 3185
#
# DİQQƏT 1: aktivasiya ReLU olduğu üçün kod mənfi ola bilməz. Bəzi
# neyronlar bütün şəkillər üçün 0 qala bilər ("ölü neyron"); bu halda kodun
# faydalı ölçüsü 49-dan az olur.
# DİQQƏT 2: 49 = 7 × 7. Bu rəqəm təsadüf ola bilər, amma kodu istəsək 7x7
# kiçik şəkil kimi də göstərə bilərik. Bu kodda isə göstərilmir, sadəcə vektordur.

d = Dense(64, activation='relu')(encoded)
# DECODER-in başlanğıcı: koddan yenidən genişlənməyə başlayır: 49 → 64.
# Forma: (49,) → (64,)
# Parametr sayı: 49 × 64 + 64 = 3200

d = Dense(28 * 28, activation='sigmoid')(d)
# Çıxış qatı: 64 → 784 (28 × 28 = 784 piksel).
# aktivasiya sigmoid: hər çıxışı 0..1 aralığına sıxır. Bu, piksel qiymətlərinin
# aralığı ilə (0..1) eynidir. ReLU olsaydı qiymətlər 1-dən böyük ola bilərdi.
# Forma: (64,) → (784,)
# Parametr sayı: 64 × 784 + 784 = 50960

decoded = Reshape((28, 28, 1))(d)
# 784 ədədlik vektoru yenidən şəkil formasına qaytarır: (784,) → (28, 28, 1).
# Beləliklə çıxışın forması girişlə EYNİ olur və ikisini müqayisə etmək olar.
# Parametr: yoxdur.

autoencoder = keras.Model(input_img, decoded, name="autoencoder")
# Modeli birləşdirir: "input_img-dən başla, decoded-də bitir".
# name="autoencoder" → modelin adı (summary-də görünür).
#
# Ümumi parametr sayı: 100480 + 8256 + 3185 + 3200 + 50960 = 166081.
# (model.summary() çağırılmayıb, çağırsaq bu rəqəmi görərdik.)
#
# DİQQƏT 1: arxitektura SİMMETRİK DEYİL. Encoder 3 qatdır (128 → 64 → 49),
# decoder isə 2 qat (64 → 784). Decoder daha zəifdir. Tam simmetrik üçün
# 49 → 64 → 128 → 784 olardı. Bu pis deyil, sadəcə qeyd edirik.
# DİQQƏT 2: ayrıca "encoder" modeli düzəldilməyib. Yəni bu kodda gizli
# kodu (encoded) tək başına çıxarmaq mümkün deyil. İstəsək:
#   encoder = keras.Model(input_img, encoded)
# yazıb encoder.predict(şəkillər) ilə 49 ədədlik kodları ala bilərik.

autoencoder.compile(optimizer='adam', loss='mean_squared_error')
# loss='mean_squared_error' (MSE) → orta kvadratik səhv. Hər pikselin
# fərqinin kvadratı götürülür və ortalaması tapılır:
#   MSE = ortalama( (orijinal piksel - bərpa olunmuş piksel)² )
# Sıfır → bərpa tam dəqiqdir. Böyük ədəd → bərpa pisdir.
#
# optimizer='adam' → defolt öyrənmə sürəti (0.001) ilə Adam.
#
# DİQQƏT: metrics yoxdur. Accuracy burada mənasızdır (piksel dəqiq
# düşdü/düşmədi demək olmaz). Yalnız loss izlənir.
# Alternativ loss: 'binary_crossentropy' də MNIST avtoenkoderlərində tez-tez
# işlədilir (sigmoid çıxışı ilə yaxşı uyğunlaşır).


# ---------------------------------------------------------------------
# 3) ÖYRƏTMƏ
# ---------------------------------------------------------------------

autoencoder.fit(x_train, x_train,
                epochs=20,
                batch_size=batch_size,
                shuffle=True)
# Öyrətmə. BAXIN: fit(X, Y) yerinə fit(x_train, x_train) yazılıb —
# giriş də, cavab da eyni şəkildir. Model "özünü təkrarla" tapşırığını öyrənir.
#
# epochs=20          → bütün 60000 şəkil 20 dəfə keçilir.
# batch_size=100     → hər addımda 100 şəkil → epoch-da 600 addım,
#                      cəmi 20 × 600 = 12000 çəki yeniləməsi.
# shuffle=True       → hər epoch-da şəkillərin sırası qarışdırılır
#                      (bu zaten fit-in defoltudur, açıq yazılıb).
#
# NƏ GÖZLƏMƏK LAZIMDIR? Loss ilk epoch-larda tez düşür, sonra yavaşlayır.
# Təxminən 0.01 civarında sabitləşməsi gözlənilir (dəqiq rəqəm hər işlətmədə
# dəyişir). Bu, bərpanın kifayət qədər yaxşı, amma bir qədər bulanıq
# olduğunu göstərir. Sıfıra düşməyəcək, çünki 49 ədədə sığmayan detallar itir.
#
# DİQQƏT: validation_data yoxdur. Test datasında itkini də izləmək üçün:
#   autoencoder.fit(..., validation_data=(x_test, x_test))


# ---------------------------------------------------------------------
# 4) NƏTİCƏNİN YOXLANMASI: ORİJİNAL vs BƏRPA
# ---------------------------------------------------------------------

n = 10
# Neçə şəkil göstərəcəyik.

imgs = x_test[:n]
# Testdən ilk 10 şəkil. Forma: (10, 28, 28, 1).
# Bu şəkillər modelə öyrətmə zamanı heç vaxt verilməyib, yəni model onları ilk dəfə görür.

decoded_imgs = autoencoder.predict(x_test[:n], batch_size=n)
# Modelin bu 10 şəkli sıxıb yenidən bərpa etməsi. Forma: (10, 28, 28, 1).
# Hər şəkil əvvəlcə 49 ədədə çevrilir, sonra yenidən 784 pikselə qaytarılır.

plt.figure(figsize=(n, 2))
# Yeni şəkil pəncərəsi: eni 10 düym, hündürlüyü 2 düym (hər rəqəmə təxminən 1 düym).

for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    # 2 sətirli, n sütunlu cədvəl düzəldir. i + 1 → ÜST sətirin i-ci xanası.
    # (Subplot nömrələri 1-dən başlayır, ona görə +1.)
    plt.imshow(imgs[i].squeeze(), cmap='gray')
    # ÜST sətirdə ORİJİNAL şəkil çəkilir.
    # .squeeze() → ölçüsü 1 olan oxları silir: (28, 28, 1) → (28, 28).
    # imshow tək kanallı şəkil üçün 2 ölçülü massiv gözləyir.
    # cmap='gray' → boz ton rəng sxemi.
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    # Oxlardakı rəqəmləri və cizgiləri gizlədir ki, yalnız şəkil görünsün.

    ax2 = plt.subplot(2, n, i + n + 1)
    # i + n + 1 → ALT sətirin i-ci xanası (n xana üstdə olduğu üçün n əlavə olunur).
    plt.imshow(decoded_imgs[i].squeeze(), cmap='gray')
    # ALT sətirdə modelin BƏRPA ETDİYİ şəkil çəkilir.
    ax2.get_xaxis().set_visible(False)
    ax2.get_yaxis().set_visible(False)

plt.show()
# Şəkli göstərir. Nəticə: yuxarıda orijinal rəqəmlər, aşağıda onların bərpası.
# Bərpa orijinala bənzəməlidir, amma adətən bir az bulanıq və dəqiq olmayan olur.
# Bu, 49 ədədlik koda sıxmanın "qiymətidir".


# ---------------------------------------------------------------------
# 5) KÖMƏKÇİ FUNKSİYALAR: ŞƏKİLLƏRİ SIRA İLƏ GÖSTƏRMƏK
# ---------------------------------------------------------------------

def plot_digits(*images):
    # *images → istənilən sayda massiv qəbul edir. Hər massiv bir sətir olaraq
    # çəkilir. Məsələn plot_digits(a, b) → 2 sətir, plot_digits(a) → 1 sətir.
    # Hər massivin forması (şəkil_sayı, 28, 28, 1) və ya (şəkil_sayı, 28, 28) olmalıdır.

    images = [x.squeeze() for x in images]
    # Hər massivdən uzunluğu 1 olan ölçüləri silir: (10, 28, 28, 1) → (10, 28, 28).
    # DİQQƏT: əgər şəkil sayı da 1 olsaydı, squeeze ona aid ölçünü də silərdi
    # və forma pozulardı. Burada n=10 olduğu üçün problem yoxdur.

    n = images[0].shape[0]  # число изображений
    # (Rusca şərh: "şəkillərin sayı".)
    # Birinci massivdə neçə şəkil olduğu. Bu, sütunların sayı olacaq.
    # DİQQƏT: bu n funksiyanın daxilindəki lokal dəyişəndir, yuxarıdakı
    # qlobal n=10 ilə qarışdırmayın (bir-birinə təsir etmir).

    plt.figure(figsize=(n, len(images)))
    # Pəncərənin ölçüsü: eni = şəkil sayı qədər düym, hündürlüyü = sətir sayı qədər düym.
    for j in range(n):
        # j → sütun nömrəsi (hansı şəkil).
        for i in range(len(images)):
            # i → sətir nömrəsi (hansı massiv).
            ax = plt.subplot(len(images), n, i * n + j + 1)
            # Cədvəlin i-ci sətrinin j-ci xanasını seçir.
            # Nömrə düsturu: i * n + j + 1.
            # Məsələn, 2 sətir, n=10: i=1, j=3 → 1×10 + 3 + 1 = 14 → 2-ci sətir, 4-cü sütun.
            plt.imshow(images[i][j])
            # i-ci massivin j-ci şəklini çəkir.
            plt.gray()
            # Cari şəkil üçün rəng sxemini boz tona keçirir.
            # (Yuxarıda cmap='gray' yazırdıq, burada bu əmrlə eyni nəticə alınır.)
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)

    plt.show()


def plot_homotopy(frm, to, n=10, autoencoder=None):
    # İki şəkil arasında "keçid" yaradır və göstərir.
    # frm → başlanğıc şəkil, to → son şəkil (hər ikisi (28, 28, 1) formasında).
    # n → keçid neçə addımdan ibarət olsun.
    # autoencoder → əgər model verilərsə, hər aralıq şəkil modeldən keçirilir.
    #
    # HOMOTOPİYA nədir? Bir obyektin başqasına kəsintisiz çevrilməsi.
    # Burada ən sadə növü var: iki şəkli xətt üzrə "qarışdırmaq" (karışdırma).
    #
    # DİQQƏT: parametrin adı autoencoder-dir və qlobal modelin adı ilə eynidir.
    # Funksiya daxilində autoencoder dedikdə ötürülən arqument başa düşülür.

    z = np.zeros(([n] + list(frm.shape)))
    # Boş massiv: forma [n] + [28, 28, 1] = (10, 28, 28, 1).
    # n sayda şəkil üçün yer ayırır, hamısı sıfırla doludur.

    for i, t in enumerate(np.linspace(0., 1., n)):
        # np.linspace(0., 1., 10) → 0.0, 0.111, 0.222, ..., 1.0 (10 bərabər addım).
        # enumerate → hər addımın nömrəsini (i) və qiymətini (t) verir.
        z[i] = frm * (1 - t) + to * t  # Гомотопия по прямой
        # (Rusca şərh: "xətt üzrə homotopiya".)
        # Çəkili ortalama:
        #   t = 0   → 1.0 × frm + 0.0 × to  → tam başlanğıc şəkil
        #   t = 0.5 → 0.5 × frm + 0.5 × to  → iki şəklin yarı-yarı qarışığı
        #   t = 1   → 0.0 × frm + 1.0 × to  → tam son şəkil
        # Hər addımda frm-in payı azalır, to-nun payı artır.
        #
        # DİQQƏT: bu qarışdırma PİKSEL fəzasında aparılır (şəkillərin özü
        # qarışdırılır), gizli kod fəzasında yox.
    if autoencoder:
        # Model verilibsə (None deyilsə).
        plot_digits(autoencoder.predict(z, batch_size=n))
        # Bütün 10 qarışıq şəkil modeldən keçirilir və nəticə çəkilir.
        # Yəni model hər qarışığı "öz anladığı kimi" bərpa edir.
    else:
        plot_digits(z)
        # Model yoxdursa, qarışıq şəkillər olduğu kimi göstərilir.


# ---------------------------------------------------------------------
# 6) HOMOTOPİYANIN GÖSTƏRİLMƏSİ
# ---------------------------------------------------------------------

frm, to = x_test[y_test == 5][1:3]
# Testdən iki "5" rəqəmi seçir. Addım-addım:
#
# y_test == 5     → hər test şəkli üçün True/False massivi (rəqəmi 5-dirsə True).
# x_test[maska]   → yalnız True olan şəkilləri seçir, yəni bütün "5"-ləri.
#                   Forma: (5-lərin sayı, 28, 28, 1). MNIST test dəstində təxminən 892 ədəd.
# [1:3]           → bu "5"-lər arasından 2-ci və 3-cü şəkli götürür
#                   (indekslər 1 və 2; indeks 0 atlanır). Forma: (2, 28, 28, 1).
# frm, to = ...   → iki şəkli iki dəyişənə ayırır, hər biri (28, 28, 1).
#
# Beləliklə frm və to eyni rəqəmin (5) fərqli əl yazısı variantlarıdır.

plot_homotopy(frm, to)
# 1-Cİ NƏTİCƏ (model yoxdur): bir sətirdə 10 şəkil. Soldan sağa birinci "5"
# tədricən ikinciyə çevrilir. Ortadakı şəkillər iki şəklin üst-üstə
# düşmüş "kölgəli" qarışığı kimi görünür. Bu, sadəcə piksellərin
# qarışdırılmasıdır, neyron şəbəkənin iştirakı yoxdur.

plot_homotopy(frm, to, autoencoder=autoencoder)
# 2-Cİ NƏTİCƏ (model ilə): eyni 10 qarışıq şəkil avtoenkoderdən keçirilib.
# Müqayisə: 1-ci sətirlə 2-ci sətirə baxın. Model hər qarışığı sıxıb yenidən
# açır, ona görə ortadakı qarışıq şəkillər adətən daha "təmiz", rəqəmə oxşar
# görünür və ya model onları öyrəndiyi formalardan birinə yaxınlaşdırır.
#
# DİQQƏT (konseptual): bu kodda interpolyasiya GİRİŞ (piksel) səviyyəsində
# aparılır. Avtoenkoderin əsl gücü isə GİZLİ KOD səviyyəsində interpolyasiyadır:
#   1) hər iki şəkli kodlaşdırmaq (49 ədəd),
#   2) kodlar arasında keçid yaratmaq,
#   3) hər aralıq kodu decoder-dən keçirib şəkil almaq.
# Bunun üçün encoder və decoder ayrı-ayrı modellər kimi qurulmalıdır
# (yuxarıdakı "DİQQƏT 2" qeydinə baxın). Bu zaman keçid daha hamar və
# "real rəqəmə oxşar" olur, çünki kod fəzası şəkil fəzasından daha mənalıdır.